In [13]:
!pip install torch torchvision --quiet

In [14]:
import pandas as pd
import numpy as np
import os
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [32]:
df = pd.read_csv("dataset_with_latlon_reindexed.csv")
df.columns = df.columns.str.strip()

In [33]:
import re
import numpy as np

def convert_price(price):
    try:
        price = str(price).replace("₹", "").replace(",", "").strip().lower()
        price = re.sub(r"[a-zA-Z]+", "", price).strip()
        return float(price)
    except:
        return None

# convert
df["Price"] = df["Price"].apply(convert_price)

# remove bad rows
df = df.dropna(subset=["Price"])

# create log target
df["log_price"] = np.log1p(df["Price"])

In [34]:
print(df.columns)
print(df["log_price"].head())

Index(['Name', 'Property Title', 'Price', 'Location', 'Total_Area',
       'Price_per_SQFT', 'Description', 'Baths', 'Balcony', 'latitude',
       'longitude', 'log_price'],
      dtype='object')
0    1.095273
1    1.178655
2    0.693147
3    1.465568
4    3.891820
Name: log_price, dtype: float64


In [35]:
import re

def convert_price(price):
    try:
        price = str(price).replace("₹", "").replace(",", "").strip().lower()
        
        # remove weird text like "acs"
        price = re.sub(r"[a-zA-Z]+", "", price).strip()
        
        if "cr" in str(price).lower():
            return float(price.replace("cr", "")) * 1e7
        elif "l" in str(price).lower():
            return float(price.replace("l", "")) * 1e5
        else:
            return float(price)
    
    except:
        return None  

In [36]:
df["Price"] = df["Price"].apply(convert_price)

In [37]:
image_dir = os.path.join(os.getcwd(), "images_reindexed")

df["image_path"] = df.index.map(
    lambda x: os.path.join(image_dir, f"{x}.png")
)

In [38]:
print(df[["image_path"]].head())

                                          image_path
0  C:\Users\mrvin\anaconda3\Capstone\notebooks\im...
1  C:\Users\mrvin\anaconda3\Capstone\notebooks\im...
2  C:\Users\mrvin\anaconda3\Capstone\notebooks\im...
3  C:\Users\mrvin\anaconda3\Capstone\notebooks\im...
4  C:\Users\mrvin\anaconda3\Capstone\notebooks\im...


In [39]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [40]:
class HouseDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        img = Image.open(row["image_path"]).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        price = torch.tensor(row["log_price"], dtype=torch.float32)
        
        return img, price

In [47]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [48]:
train_dataset = HouseDataset(train_df, transform)
test_dataset = HouseDataset(test_df, transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [49]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# freeze backbone
for param in model.parameters():
    param.requires_grad = False

# train only final layer
model.fc = nn.Linear(model.fc.in_features, 1)

In [52]:
model = model.to(device)

In [53]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

In [54]:
epochs = 7

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for images, targets in train_loader:
        images = images.to(device)
        targets = targets.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, targets)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

Epoch 1, Loss: 3.251495523439659
Epoch 2, Loss: 1.9880553896610553
Epoch 3, Loss: 1.9287324544492659
Epoch 4, Loss: 1.88236272416927
Epoch 5, Loss: 1.8496304232995588
Epoch 6, Loss: 1.8231535910905063
Epoch 7, Loss: 1.803836648608302


In [55]:
model.eval()

preds = []
actuals = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        
        outputs = model(images)
        
        preds.extend(outputs.cpu().numpy())
        actuals.extend(targets.numpy())

preds = np.array(preds).flatten()
actuals = np.array(actuals).flatten()

rmse = np.sqrt(mean_squared_error(actuals, preds))
r2 = r2_score(actuals, preds)

print("IMAGE-ONLY (ResNet18)")
print("RMSE:", rmse)
print("R2:", r2)

IMAGE-ONLY (ResNet18)
RMSE: 1.3646610131127213
R2: -0.050658464431762695
